# Module 6: Urban Expansion & Road Network Analysis

**Task 6**: Assess urban expansion patterns using:
- Road network buffers (1 and 2 km)
- Multi-ring growth gradients (1, 3, 5 km from city centers)
- Distance-vs-urban-density curves
- Correlation: Has urban growth happened along major roads?

**Deliverables**:
- Road buffer stats tables
- Distance-density curves per city
- Pearson correlation results + scatter plots
- Urban growth rate tables
- Urban expansion maps

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterstats import zonal_stats
from scipy import stats as scipy_stats

from config import (
    STATES, YEARS, LULC_CLASSES, LULC_COLORS, NUM_CLASSES,
    PROCESSED_DIR, FIGURES_DIR, LULC_DIR, ROADS_DIR, BOUNDARIES_DIR,
    ROAD_BUFFER_DISTANCES_M, URBAN_RING_DISTANCES_M,
    TIER2_SCALE, PIXEL_AREA_KM2,
)
from scripts.buffer_analysis import (
    create_road_buffers, create_urban_growth_rings,
    create_distance_gradient_samples,
)
from scripts.visualization import save_figure, plot_urban_density_curve
from scripts.lulc_utils import save_composition_csv

print('Imports successful!')

## 6.1: Load Road Network

In [ ]:
roads = {}

for state_key, state_cfg in STATES.items():
    road_file = ROADS_DIR / f'{state_key}_major_roads.gpkg'
    if road_file.exists():
        roads[state_key] = gpd.read_file(road_file)
        print(f'{state_cfg["name"]}: {len(roads[state_key])} road segments loaded')
    else:
        print(f'⚠️ {road_file.name} not found — run Module 1 road download')

## 6.2: Road Network Buffers (1 km and 2 km)

In [ ]:
road_buffer_stats = {}

for state_key, state_cfg in STATES.items():
    if state_key not in roads:
        continue
    
    print(f'Creating road buffers for {state_cfg["name"]}...')
    road_rings = create_road_buffers(
        roads[state_key], state_cfg['utm_epsg'],
        distances_m=ROAD_BUFFER_DISTANCES_M
    )
    
    # Save
    outdir = PROCESSED_DIR / 'buffers'
    outdir.mkdir(parents=True, exist_ok=True)
    road_rings.to_file(outdir / f'{state_key}_road_buffers.geojson', driver='GeoJSON')
    
    # Compute built-up area in each buffer for each year
    all_rows = []
    for year in YEARS:
        raster_path = LULC_DIR / f'{state_key}_30m_{year}.tif'
        if not raster_path.exists():
            print(f'  ⚠️ {raster_path.name} not found')
            continue
        
        stats = zonal_stats(road_rings, str(raster_path), categorical=True, nodata=255)
        
        for i, (_, ring) in enumerate(road_rings.iterrows()):
            ring_stats = stats[i] if i < len(stats) else {}
            total = sum(ring_stats.values()) if ring_stats else 1
            built = ring_stats.get(6, 0)  # Class 6 = Built Area
            
            all_rows.append({
                'year': year,
                'ring_label': ring['ring_label'],
                'distance_km': ring['distance_outer_m'] / 1000,
                'built_pixels': built,
                'total_pixels': total,
                'built_pct': (built / total * 100) if total > 0 else 0,
                'built_area_km2': built * PIXEL_AREA_KM2[TIER2_SCALE],
            })
    
    if all_rows:
        buf_df = pd.DataFrame(all_rows)
        road_buffer_stats[state_key] = buf_df
        save_composition_csv(buf_df, outdir / f'{state_key}_road_buffer_buildup.csv')
        
        print(f'\n{state_cfg["name"]} — Built-up in Road Buffers:')
        display(buf_df.pivot_table(index='ring_label', columns='year',
                                   values='built_pct', fill_value=0))

## 6.3: Multi-Ring Urban Growth Gradients

In [ ]:
urban_gradient_data = {}

for state_key, state_cfg in STATES.items():
    cities = state_cfg['cities']
    all_city_data = []
    
    for city in cities:
        print(f'Creating growth rings for {city["name"]}, {state_cfg["name"]}...')
        
        # Create concentric rings: 1, 3, 5 km
        rings = create_urban_growth_rings(
            city, state_cfg['utm_epsg'],
            distances_m=URBAN_RING_DISTANCES_M
        )
        
        for year in YEARS:
            raster_path = LULC_DIR / f'{state_key}_30m_{year}.tif'
            if not raster_path.exists():
                continue
            
            stats = zonal_stats(rings, str(raster_path), categorical=True, nodata=255)
            
            for i, (_, ring) in enumerate(rings.iterrows()):
                ring_stats = stats[i] if i < len(stats) else {}
                total = sum(ring_stats.values()) if ring_stats else 1
                built = ring_stats.get(6, 0)
                
                all_city_data.append({
                    'city': city['name'],
                    'year': year,
                    'ring_label': ring['ring_label'],
                    'distance_km': ring['distance_outer_m'] / 1000,
                    'built_pct': (built / total * 100) if total > 0 else 0,
                    'built_area_km2': built * PIXEL_AREA_KM2[TIER2_SCALE],
                })
    
    if all_city_data:
        gradient_df = pd.DataFrame(all_city_data)
        urban_gradient_data[state_key] = gradient_df
        save_composition_csv(
            gradient_df,
            PROCESSED_DIR / 'buffers' / f'{state_key}_urban_gradients.csv'
        )
        display(gradient_df.head(15))

## 6.4: Distance vs. Urban Density Curves

In [ ]:
for state_key, state_cfg in STATES.items():
    if state_key not in urban_gradient_data:
        continue
    
    gradient_df = urban_gradient_data[state_key]
    
    for city_name in gradient_df['city'].unique():
        city_data = gradient_df[gradient_df['city'] == city_name]
        plot_urban_density_curve(city_data, city_name, state_cfg['name'])

print('✅ Distance-density curves generated.')

## 6.5: Urban Growth Rate

In [ ]:
for state_key, state_cfg in STATES.items():
    if state_key not in urban_gradient_data:
        continue
    
    gradient_df = urban_gradient_data[state_key]
    
    print(f'\n{state_cfg["name"]} — Urban Growth Rates:')
    print('=' * 60)
    
    for city_name in gradient_df['city'].unique():
        city_data = gradient_df[gradient_df['city'] == city_name]
        print(f'\n  {city_name}:')
        
        for ring_label in city_data['ring_label'].unique():
            ring_data = city_data[city_data['ring_label'] == ring_label]
            
            pct_2016 = ring_data[ring_data['year'] == 2016]['built_pct'].values
            pct_2025 = ring_data[ring_data['year'] == 2025]['built_pct'].values
            
            if len(pct_2016) > 0 and len(pct_2025) > 0:
                growth = pct_2025[0] - pct_2016[0]
                rate = growth / 9  # Annual rate over 9 years
                print(f'    {ring_label}: {pct_2016[0]:.1f}% → {pct_2025[0]:.1f}% '
                      f'(Δ={growth:+.1f}%, {rate:+.2f}%/yr)')

## 6.6: Correlation — Urban Growth vs. Road Proximity

In [ ]:
for state_key, state_cfg in STATES.items():
    if state_key not in road_buffer_stats:
        continue
    
    buf_df = road_buffer_stats[state_key]
    
    # For each year, compute correlation between distance and built-up %
    fig, axes = plt.subplots(1, len(YEARS), figsize=(6 * len(YEARS), 5))
    if len(YEARS) == 1:
        axes = [axes]
    
    print(f'\n{state_cfg["name"]} — Road Proximity vs. Built-up Correlation:')
    
    for ax, year in zip(axes, YEARS):
        year_data = buf_df[buf_df['year'] == year]
        
        if len(year_data) >= 2:
            x = year_data['distance_km'].values
            y = year_data['built_pct'].values
            
            r, p = scipy_stats.pearsonr(x, y) if len(x) > 2 else (0, 1)
            
            ax.scatter(x, y, s=100, c='#C4281B', zorder=5)
            ax.set_xlabel('Buffer Distance (km)')
            ax.set_ylabel('Built-up (%)')
            ax.set_title(f'{year} (r={r:.3f}, p={p:.3f})')
            ax.grid(True, alpha=0.3)
            
            # Trend line
            if len(x) > 1:
                z = np.polyfit(x, y, 1)
                ax.plot(np.sort(x), np.polyval(z, np.sort(x)), '--', color='grey')
            
            print(f'  {year}: r={r:.3f}, p={p:.3f}')
    
    fig.suptitle(f'Road Proximity vs Built-up — {state_cfg["name"]}', fontsize=13)
    fig.tight_layout()
    save_figure(fig, f'{state_key}_road_correlation.png', 'urban_gradient')
    plt.show()

print('\n✅ Module 6 complete.')

---
## Summary
- ✅ Road buffers (1 km, 2 km) with built-up area tracking
- ✅ Multi-ring urban growth gradients (1, 3, 5 km) for all cities
- ✅ Distance-vs-urban-density curves
- ✅ Urban growth rates (core vs. periphery)
- ✅ Pearson correlation: road proximity vs. built-up density

**Next**: `07_hotspot_analysis.ipynb`